# 05. Advanced Analysis

Advanced analysis is used to uncover deeper operational patterns and identify combinations of factors that contribute to delivery delays.

### Key Business Questions

- Which combination of operational factors creates the highest delivery risk?
- Which delivery conditions are most likely to result in late deliveries?
- Can high-risk delivery situations be identified using multiple factors together?
- Which factors should be prioritized for operational improvement?

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/food_delivery_cleaned.csv")

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


## 1. High-Risk Delivery Conditions

Analyze combinations of operational factors to identify delivery situations with a high probability of late delivery.

### Key Business Questions

- Which combination of traffic and weather conditions has the highest late-delivery risk?
- How does multiple-delivery workload affect high-risk conditions?
- Which operational situations should receive the highest priority?
- Can these combinations be used to improve delivery planning?

In [3]:
df["Late_Delivery"] = np.where(
    df["Time_taken(min)"] > 30,
    "Late",
    "On Time"
)

risk_analysis = (
    df.groupby(
        ["Road_traffic_density", "Weatherconditions"]
    )
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

risk_analysis.head(10)

,Road_traffic_density,Weatherconditions,Orders,Avg_Delivery_Time,Late_Rate
7,Jam,conditions Fog,2429,36.806916,76.080692
6,Jam,conditions Cloudy,2349,36.689655,74.840358
18,Medium,conditions Cloudy,1838,28.483134,48.313384
19,Medium,conditions Fog,1852,28.044816,45.896328
11,Jam,conditions Windy,2351,30.219056,45.555083
8,Jam,conditions Sandstorms,2399,30.018758,44.518549
9,Jam,conditions Stormy,2323,29.850194,43.349118
3,High,conditions Stormy,733,27.845839,38.744884
2,High,conditions Sandstorms,701,27.711840,38.373752
0,High,conditions Cloudy,744,28.940860,37.634409


### Business Interpretation

The analysis shows that delivery risk increases substantially when unfavorable weather conditions occur together with heavy traffic.

The highest-risk combination is **Fog + Jam**, with an average delivery time of **36.81 minutes** and a late-delivery rate of **76.08%** across 2,429 orders. **Cloudy + Jam** is similarly high-risk, with a late-delivery rate of **74.84%** and an average delivery time of **36.69 minutes**.

Medium traffic combined with Cloudy or Fog conditions also creates elevated risk, with late-delivery rates of **48.31%** and **45.90%**, respectively.

These results show that operational factors should not be considered independently. The combination of traffic and weather can create substantially higher delivery risk than either factor alone.

**Business Insight:** During high-risk combinations such as Fog + Jam and Cloudy + Jam, the company should consider proactive ETA adjustments, additional delivery capacity, improved route planning, and reduced order batching to minimize late deliveries.

## 2. Multiple Deliveries × Traffic

Analyze how delivery workload and traffic conditions interact to affect delivery performance.

### Key Business Questions

- Does multiple-delivery workload become more problematic under heavy traffic?
- Which combination of traffic and delivery workload has the highest late-delivery risk?
- Should multiple deliveries be restricted during high-traffic conditions?
- Which operational combinations require the most attention?

In [6]:
multiple_traffic_analysis = (
    df.groupby(["Road_traffic_density", "multiple_deliveries"]
)
.agg(
    orders = ("Time_taken(min)", "count"),
    Avg_Delivery_Time = ("Time_taken(min)", "mean"),
    Late_Rate = ("Late_Delivery", lambda x: (x == "Late").mean()*100)
)
.reset_index()
.sort_values("Late_Rate", ascending=False)
)

multiple_traffic_analysis.head(15)

,Road_traffic_density,multiple_deliveries,orders,Avg_Delivery_Time,Late_Rate
2,High,2,175,39.760000,100.000000
7,Jam,2,1155,41.464935,100.000000
3,High,3,35,47.828571,100.000000
8,Jam,3,264,48.185606,100.000000
22,NaN,2,28,40.071429,100.000000
17,Medium,2,454,38.863436,100.000000
18,Medium,3,50,46.180000,100.000000
13,Low,3,5,43.600000,100.000000
12,Low,2,173,38.653179,100.000000
23,NaN,3,7,48.714286,100.000000


### Business Interpretation

The analysis shows that delivery workload becomes significantly more problematic when combined with traffic congestion.

The **Jam + 2 deliveries** combination is particularly concerning, with 1,155 orders, an average delivery time of **41.46 minutes**, and a **100% late-delivery rate**. The **Jam + 3 deliveries** group also shows a 100% late rate with an average delivery time of **48.19 minutes**, although its smaller sample size requires more cautious interpretation.

Multiple deliveries under High and Medium traffic also show 100% late rates for groups with 2 or more deliveries. In comparison, Jam traffic with a single delivery has a much larger sample of 8,848 orders and a late rate of **49.19%**.

These results indicate that delivery workload and traffic can interact to create substantially higher operational risk.

**Business Insight:** Multiple-order assignments should be carefully controlled during periods of heavy traffic. Reducing order batching during peak congestion could help prevent severe delivery delays.

## 3. Delivery Partner Performance Under Traffic

Analyze whether delivery partner performance changes under different traffic conditions.

### Key Business Questions

- Which delivery partners perform best under heavy traffic?
- Which partners have the highest delivery times under difficult traffic conditions?
- Are some delivery partners more consistent across traffic conditions?
- Can partner performance be considered when assigning difficult deliveries?

In [7]:
partner_traffic_analysis = (
    df.groupby(
        ["Delivery_person_ID", "Road_traffic_density"]
    )
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
)

partner_traffic_analysis = partner_traffic_analysis[
    partner_traffic_analysis["Orders"] >= 20
].sort_values(
    ["Road_traffic_density", "Avg_Delivery_Time"]
)

partner_traffic_analysis.head(20)

,Delivery_person_ID,Road_traffic_density,Orders,Avg_Delivery_Time,Late_Rate
3702,MUMRES04DEL01,High,21,27.761905,33.333333
2130,HYDRES06DEL02,Jam,20,24.300000,20.000000
1267,CHENRES12DEL03,Jam,22,25.363636,22.727273
2447,INDORES12DEL03,Jam,22,26.318182,36.363636
2686,JAPRES11DEL03,Jam,24,26.375000,25.000000
3734,MUMRES06DEL03,Jam,25,26.480000,36.000000
3794,MUMRES12DEL03,Jam,21,26.619048,19.047619
695,BANGRES03DEL03,Jam,27,26.629630,25.925926
4806,SURRES12DEL03,Jam,20,26.900000,40.000000
4682,SURRES01DEL03,Jam,21,26.952381,23.809524


### Business Interpretation

The analysis shows that delivery partner performance varies under different traffic conditions. Even under Jam traffic, some delivery partners maintain lower average delivery times and lower late-delivery rates than others.

For example, `HYDRES06DEL02` recorded an average delivery time of **24.30 minutes** with a **20% late-delivery rate** across 20 orders, while `SURRES12DEL03` recorded an average of **26.90 minutes** with a **40% late-delivery rate** across the same minimum sample threshold.

These differences suggest that some delivery partners may perform more consistently under difficult operating conditions. However, the individual partner groups in this analysis have relatively small sample sizes, so the results should be treated as directional rather than definitive.

**Business Insight:** Partner performance under difficult traffic conditions can be used as an additional input when assigning time-sensitive or high-risk deliveries, but decisions should be based on larger historical samples rather than a small number of orders.

## 4. Combined High-Risk Delivery Segment

Combine multiple operational risk factors to identify delivery situations with the highest overall delay risk.

### Key Business Questions

- Which combination of traffic, weather, and workload creates the highest delivery risk?
- How does the high-risk segment compare with other deliveries?
- What proportion of orders falls into the high-risk segment?
- Can this segment be used for proactive delivery planning?

In [9]:
df["multiple_deliveries"] = pd.to_numeric(
    df["multiple_deliveries"],
    errors="coerce"
)

In [20]:
df["High_Risk_Condition"] = np.where(
    (df["Road_traffic_density"].isin(["Jam", "High"])) &
    (df["Weatherconditions"].isin(["conditions Fog", "conditions Cloudy"])) &
    (df["multiple_deliveries"] >= 2),
    "High Risk",
    "Other"
)

high_risk_summary = (
    df.groupby("High_Risk_Condition")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
)

high_risk_summary["Order_Share"] = (
    high_risk_summary["Orders"] / len(df) * 100
)

high_risk_summary

,High_Risk_Condition,Orders,Avg_Delivery_Time,Late_Rate,Order_Share
0,High Risk,843,42.563464,100.00000,1.848968
1,Other,44750,25.988134,28.52067,98.151032


### Section 4 Interpretation

**1. Which combination creates the highest delivery risk?**  
The **High Risk** segment — Jam/High traffic + Fog/Cloudy weather + 2+ deliveries — has an average delivery time of **42.56 minutes** and a **100% late-delivery rate**.

**2. How does it compare with other deliveries?**  
The remaining **44,750 orders** have an average delivery time of **25.99 minutes** and a **28.52% late rate**. The high-risk segment is therefore about **16.57 minutes slower** with a substantially higher late-delivery rate.

**3. What proportion of orders falls into this segment?**  
The high-risk segment contains **843 orders**, representing **1.85% of all deliveries**.

**4. Can it be used for proactive planning?**  
Yes. Although this segment represents only **1.85% of orders**, its very high delay risk makes it useful for proactive ETA adjustments, reducing multiple-order assignments, and allocating additional delivery capacity.

### Business Insight

A small but critical group of **843 deliveries (1.85%)** experiences an average delivery time of **42.56 minutes** and a **100% late-delivery rate** when high traffic, poor weather, and multiple deliveries occur together. This suggests that delivery operations should proactively identify these conditions and reduce workload or adjust ETAs before delays occur.

**Note:** The 100% late rate is based on historical data and should be treated as a high-risk signal rather than a guarantee for future deliveries.

## 5. Delivery Risk Scoring

Create a simple risk score for each delivery based on multiple operational factors and identify which factors contribute most to overall delivery risk.

### Key Business Questions

- Which deliveries have the highest overall risk scores?
- How does delivery time and late rate change across risk levels?
- Which operational factors contribute most to delivery risk?
- Can risk scoring help prioritize deliveries for intervention?

In [21]:
df["Risk_Score"] = 0

# Traffic risk
df["Risk_Score"] += np.where(
    df["Road_traffic_density"] == "Jam", 2,
    np.where(df["Road_traffic_density"] == "High", 1, 0)
)

# Weather risk
df["Risk_Score"] += np.where(
    df["Weatherconditions"].isin(
        ["conditions Fog", "conditions Cloudy"]
    ),
    2,
    0
)

# Multiple-delivery risk
df["Risk_Score"] += np.where(
    df["multiple_deliveries"] >= 2,
    2,
    np.where(df["multiple_deliveries"] == 1, 1, 0)
)

# Convert score into risk levels
df["Risk_Level"] = pd.cut(
    df["Risk_Score"],
    bins=[-1, 2, 4, 6],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

risk_score_analysis = (
    df.groupby("Risk_Level", observed=True)
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100),
        Avg_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

risk_score_analysis

,Risk_Level,Orders,Avg_Delivery_Time,Late_Rate,Avg_Risk_Score
0,Low Risk,26852,23.273760,16.609564,1.051095
1,Medium Risk,14980,28.936048,41.241656,3.212150
2,High Risk,3761,37.341399,78.915182,5.211380


### Section 5 Interpretation

**1. Which deliveries have the highest overall risk scores?**  
The **High Risk** group has an average risk score of **5.21**, indicating that these deliveries are exposed to multiple operational risk factors simultaneously.

**2. How does delivery performance change across risk levels?**  
Delivery performance deteriorates consistently as risk increases:

- **Low Risk:** 23.27 minutes average delivery time and **16.61%** late rate.
- **Medium Risk:** 28.94 minutes average delivery time and **41.24%** late rate.
- **High Risk:** 37.34 minutes average delivery time and **78.92%** late rate.

**3. Which factors contribute most to delivery risk?**  
The risk score combines **traffic, weather, and multiple-delivery workload**. The increasing delivery time and late rate across risk levels indicate that the combination of these factors is strongly associated with delivery delays.

**4. Can risk scoring help prioritize deliveries for intervention?**  
Yes. Deliveries classified as **High Risk** can be prioritized for proactive ETA adjustments, route optimization, and workload reduction.

### Business Insight

The risk-scoring approach effectively separates deliveries into distinct performance groups. **High-risk deliveries take approximately 14 minutes longer than low-risk deliveries and have a late rate of 78.92% compared with 16.61% for low-risk deliveries.** This provides a practical framework for identifying deliveries that require intervention before delays occur.

**Note:** The risk score is a rule-based analytical framework created for this project; it indicates relative risk rather than predicting future delays with a trained machine-learning model.

## 6. Operational Performance Segmentation

Segment deliveries into broader operational profiles to identify the most and least efficient delivery conditions.

### Key Business Questions

- Which operational profiles have the best and worst delivery performance?
- Which profiles combine high delivery times with high late-delivery rates?
- How large is each operational profile?
- Which profile should be prioritized for operational improvement?

In [23]:
# Create broader operational profiles
df["Operational_Profile"] = (
    df["Road_traffic_density"].fillna("Unknown")
    + " Traffic | "
    + df["Weatherconditions"]
        .fillna("Unknown")
        .str.replace("conditions ", "", regex=False)
    + " Weather | "
    + np.where(
        df["multiple_deliveries"] >= 2,
        "2+ Deliveries",
        np.where(
            df["multiple_deliveries"] == 1,
            "1 Delivery",
            "0 Deliveries"
        )
    )
)

# Analyze operational profiles
profile_analysis = (
    df.groupby("Operational_Profile")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
)

# Calculate order share
profile_analysis["Order_Share"] = (
    profile_analysis["Orders"] / len(df) * 100
)

# Focus on profiles with sufficient volume
profile_analysis = (
    profile_analysis[
        profile_analysis["Orders"] >= 100
    ]
    .sort_values(
        ["Late_Rate", "Orders"],
        ascending=[False, False]
    )
)

profile_analysis = profile_analysis.reset_index(drop=True)

profile_analysis.head(15)

,Operational_Profile,Orders,Avg_Delivery_Time,Late_Rate,Order_Share
0,Jam Traffic | Fog Weather | 2+ Deliveries,408,43.007353,100.000000,0.894874
1,Jam Traffic | Cloudy Weather | 2+ Deliveries,387,42.901809,100.000000,0.848815
2,Jam Traffic | Windy Weather | 2+ Deliveries,191,41.947644,100.000000,0.418924
3,Jam Traffic | Stormy Weather | 2+ Deliveries,181,42.270718,100.000000,0.396991
4,Jam Traffic | Sandstorms Weather | 2+ Deliveries,180,42.050000,100.000000,0.394797
5,Medium Traffic | Cloudy Weather | 2+ Deliveries,153,40.130719,100.000000,0.335578
6,Medium Traffic | Fog Weather | 2+ Deliveries,134,39.679104,100.000000,0.293905
7,Jam Traffic | Fog Weather | 1 Delivery,1505,35.873090,73.687708,3.300945
8,Jam Traffic | Cloudy Weather | 1 Delivery,1413,35.789809,71.903751,3.099160
9,Jam Traffic | Cloudy Weather | 0 Deliveries,549,34.626594,64.663024,1.204132


### Section 6 Interpretation

**1. Which operational profiles have the worst performance?**  
The highest-risk profiles are combinations of **Jam traffic + poor weather + 2+ deliveries**. These profiles all have a **100% late-delivery rate** and average delivery times above **40 minutes**.

The most significant examples are:
- Jam + Fog + 2+ deliveries: **43.01 minutes**, **100% late**, 408 orders.
- Jam + Cloudy + 2+ deliveries: **42.90 minutes**, **100% late**, 387 orders.
- Jam + Windy + 2+ deliveries: **41.95 minutes**, **100% late**, 191 orders.

**2. Which profiles combine high delivery time with high late-delivery rates?**  
Jam traffic combined with Fog or Cloudy weather creates particularly severe delays even with only **1 delivery**:
- Jam + Fog + 1 delivery: **35.87 minutes**, **73.69% late**.
- Jam + Cloudy + 1 delivery: **35.79 minutes**, **71.90% late**.

This shows that heavy traffic and poor weather remain major risks even without multiple deliveries.

**3. How large are the major high-risk profiles?**  
The largest high-risk profile is **Jam + Fog + 1 delivery**, with **1,505 orders**, representing **3.30%** of all deliveries.

The Jam + Cloudy + 1 delivery profile contains **1,413 orders**, representing **3.10%** of all deliveries.

**4. Which profile should be prioritized for improvement?**  
The highest priority should be given to **Jam traffic + Fog/Cloudy weather + 2+ deliveries**, because these conditions consistently produce delivery times above 40 minutes and a 100% historical late-delivery rate.

### Business Insight

Delivery performance is strongly affected by the **combination of traffic, weather, and delivery workload** rather than any single factor alone. The most critical operational profile is **heavy traffic + poor weather + multiple deliveries**, where average delivery times exceed 40 minutes and every observed delivery was late.

The business should consider **reducing multiple-order assignments during Jam traffic and poor weather**, while proactively adjusting ETAs for these operational profiles.